# lm_forge — Pretrain 85M Base Model (Colab T4)

This notebook pretrains an 85M parameter model on:
- **Phase 1:** `HuggingFaceFW/fineweb-edu` (general knowledge)
- **Phase 2:** `bigcode/the-stack-v2` (code)

**Hardware:** Google Colab T4 (16GB VRAM)  
**Precision:** FP16 with GradScaler  
**Auto-resume:** Re-run the same cell to continue from latest checkpoint

## 1. Setup

In [ ]:
# Clone repo (skip if already cloned)
import os
if not os.path.exists('lm_forge'):
    !git clone https://github.com/YOUR_USERNAME/lm_forge.git
%cd lm_forge

In [ ]:
# Install dependencies
!pip install -q torch transformers datasets huggingface_hub safetensors accelerate

In [ ]:
# Verify GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU. Training will be very slow on CPU.")

## 2. Smoke Test (Optional)

Run this to verify everything works before committing to a long training run.

In [ ]:
!python experiments/pretrain_base/train.py --phase 1 --smoke --steps 50

## 3. Phase 1 — General Knowledge (fineweb-edu)

Trains for ~38,000 steps (~10B tokens). Each Colab session lasts ~4-6 hours.

**Just re-run this cell if your session disconnects.** It auto-resumes from the latest checkpoint.

In [ ]:
# Phase 1 training (auto-resumes from latest checkpoint)
!python experiments/pretrain_base/train.py --phase 1 --t4

### Check Phase 1 Progress

In [ ]:
# Check latest checkpoint
import glob
checkpoints = sorted(glob.glob('checkpoints/pretrain_phase1/checkpoint-*'),
                     key=lambda p: int(p.split('-')[-1]))
if checkpoints:
    print(f"Latest checkpoint: {checkpoints[-1]}")
    import json
    state = json.load(open(f'{checkpoints[-1]}/trainer_state.json'))
    print(f"Global step: {state['global_step']}")
    print(f"Best loss: {state.get('best_metric', 'N/A')}")
    steps_remaining = 38000 - state['global_step']
    print(f"Steps remaining: {steps_remaining:,}")
else:
    print("No checkpoints found yet.")

### Quick Generation Test (Phase 1)

In [ ]:
from transformers import pipeline
import torch

ckpt_path = 'checkpoints/pretrain_phase1/final'
if not os.path.exists(ckpt_path):
    # Use latest checkpoint if final doesn't exist
    checkpoints = sorted(glob.glob('checkpoints/pretrain_phase1/checkpoint-*'),
                         key=lambda p: int(p.split('-')[-1]))
    ckpt_path = checkpoints[-1] if checkpoints else None

if ckpt_path:
    gen = pipeline('text-generation', model=ckpt_path, device=0 if torch.cuda.is_available() else -1)
    prompts = [
        "The history of science begins with",
        "In mathematics, the concept of",
        "Water is composed of",
    ]
    for p in prompts:
        out = gen(p, max_new_tokens=50, do_sample=True, temperature=0.8)
        print(f"Prompt: {p}")
        print(f"Output: {out[0]['generated_text'][:150]}\n")
else:
    print("No checkpoint available. Run Phase 1 first.")

## 4. Phase 2 — Code (the-stack-v2)

Run this **after Phase 1 is complete.**

Uses a lower learning rate (1e-4) to preserve general knowledge while learning code.

In [ ]:
# Phase 2 training (auto-resumes from Phase 1 final checkpoint)
!python experiments/pretrain_base/train.py --phase 2 \
    --resume checkpoints/pretrain_phase1/final \
    --t4

In [ ]:
# Check Phase 2 progress
checkpoints = sorted(glob.glob('checkpoints/pretrain_phase2/checkpoint-*'),
                     key=lambda p: int(p.split('-')[-1]))
if checkpoints:
    print(f"Latest checkpoint: {checkpoints[-1]}")
    state = json.load(open(f'{checkpoints[-1]}/trainer_state.json'))
    print(f"Global step: {state['global_step']}")
else:
    print("No Phase 2 checkpoints yet.")

### Quick Generation Test (Phase 2)

In [ ]:
ckpt_path = 'checkpoints/pretrain_phase2/final'
if not os.path.exists(ckpt_path):
    checkpoints = sorted(glob.glob('checkpoints/pretrain_phase2/checkpoint-*'),
                         key=lambda p: int(p.split('-')[-1]))
    ckpt_path = checkpoints[-1] if checkpoints else None

if ckpt_path:
    gen = pipeline('text-generation', model=ckpt_path, device=0 if torch.cuda.is_available() else -1)
    prompts = [
        "def fibonacci(n):",
        "import os\n\ndef list_files(",
        "class DataProcessor:\n    def __init__(self):",
    ]
    for p in prompts:
        out = gen(p, max_new_tokens=80, do_sample=True, temperature=0.7)
        print(f"Prompt: {p}")
        print(f"Output:\n{out[0]['generated_text']}\n")
else:
    print("No Phase 2 checkpoint. Complete Phase 2 first.")

## 5. Save to Google Drive (Optional)

Backup checkpoints to Drive so you don't lose them when Colab resets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Backup final checkpoints
!cp -r checkpoints/pretrain_phase1/final /content/drive/MyDrive/lm_forge_pretrain/phase1_final
!cp -r checkpoints/pretrain_phase2/final /content/drive/MyDrive/lm_forge_pretrain/phase2_final
print("Checkpoints backed up to Drive.")

In [ ]:
# Restore from Drive (if starting a new session)
# !cp -r /content/drive/MyDrive/lm_forge_pretrain/phase1_final checkpoints/pretrain_phase1/final
# !cp -r /content/drive/MyDrive/lm_forge_pretrain/phase2_final checkpoints/pretrain_phase2/final

## 6. Next Steps

After pretraining completes, use `checkpoints/pretrain_phase2/final` as the starting point for PS1 fine-tuning.

```bash
# In powershell_gen experiment, modify to load from pretrained:
model = HFCausalLM.from_pretrained('checkpoints/pretrain_phase2/final')
```